# Import Statements

In [51]:
import os
import glob
import shutil
import cv2
from PIL import Image
from matplotlib import pyplot as plt
import pandas as pd
import random
from ultralytics import YOLO

# Check and Set Working Directory

In [2]:
pwd

'/home/exh4748/ProjectTortoise/Beta'

In [3]:
# Setting base directory path
base_dir = "/home/exh4748/ProjectTortoise/ObjectData/data2"
os.chdir(base_dir)
base_dir = os.getcwd()

In [4]:
# List the contents of train, val, and test folders
train_contents = os.listdir(os.path.join(base_dir, "train"))
val_contents = os.listdir(os.path.join(base_dir, "val"))
test_contents = os.listdir(os.path.join(base_dir, "test"))

print("Train Folder Contents:", train_contents)
print("Validation Folder Contents:", val_contents)
print("Test Folder Contents:", test_contents)

Train Folder Contents: ['wallet', 'watch', 'cellphone']
Validation Folder Contents: ['wallet', 'watch', 'cellphone']
Test Folder Contents: ['wallet', 'watch', 'cellphone']


In [7]:
# Function to print directory structure
def print_directory_structure(base_path):
    for root, dirs, files in os.walk(base_path):
        level = root.replace(base_path, "").count(os.sep)
        indent = " " * 4 * level
        print(f"{indent}{os.path.basename(root)}/")
        sub_indent = " " * 4 * (level + 1)
        for f in files[:5]:  # Print only first 5 files for brevity
            print(f"{sub_indent}{f}")
        if len(files) > 5:
            print(f"{sub_indent}... ({len(files)} files)")

print("📂 Checking dataset structure...")
print_directory_structure(base_dir)

📂 Checking dataset structure...
data2/
    train/
        wallet/
            aug_20871_02NYXWWEYUN4.jpg
            aug_90771_INQ8R7H52NW5.jpg
            aug_25679_B5NAR2N1WXD6.jpg
            aug_70094_LQP5BEN7HVJ9.jpg
            aug_39208_SKUE2L7C7CDP.jpg
            ... (64014 files)
        watch/
            aug_50499_012_0bef5f6f.jpg
            aug_34910_079_fe404f1e.jpg
            aug_31200_067_da385cad.jpg
            aug_86031_160_02459f6d.jpg
            aug_33236_093_d34565e6.jpg
            ... (63973 files)
        cellphone/
            aug_92117_3RIPGFWFSZEP.jpg
            aug_49101_Z278HA5JIMGW.jpg
            aug_71117_GWANTZJA1220.jpg
            aug_97248_6PRC2GHAHE80.jpg
            aug_69231_XXT8A7I263G2.jpg
            ... (63935 files)
    test/
        wallet/
            aug_46864_HA771L5SYFX7.jpg
            aug_66729_Z1SNN01QXALX.jpg
            aug_66863_1H10YMAMQ0GK.jpg
            aug_24490_QP55Z1M49YGE.jpg
            aug_59500_J8BUXLDGBIFY.jpg
    

# Setting Up Dataset For Model

In [5]:
import tensorflow as tf

# Check if any GPU is detected
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  4


In [6]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Set TensorFlow to use only the first GPU
        tf.config.set_visible_devices(gpus[0], 'GPU')
        print(f"Using GPU: {gpus[0]}")
    except RuntimeError as e:
        print(e)

Using GPU: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [11]:
original_dataset_path = "/home/exh4748/ProjectTortoise/ObjectData/data2"
new_dataset_path = "/home/exh4748/ProjectTortoise/Data"

# Define dataset splits
splits = ["train", "val", "test"]
classes = ["wallet", "watch", "cellphone"]

# Create the new structured dataset directories
for cls in classes:
    for split in splits:
        os.makedirs(os.path.join(new_dataset_path, cls, "images", split), exist_ok=True)

# Move images to new structure
for split in splits:
    for cls in classes:
        old_class_path = os.path.join(original_dataset_path, split, cls)  # Old path
        new_class_path = os.path.join(new_dataset_path, cls, "images", split)  # New path

        if not os.path.exists(old_class_path):
            print(f"Skipping {old_class_path}, folder not found.")
            continue

        images = [f for f in os.listdir(old_class_path) if f.lower().endswith((".jpg", ".png", ".jpeg"))]

        if not images:
            print(f"No images found in {old_class_path}, skipping.")
            continue  # Skip empty folders

        # Move images
        for img in images:
            shutil.move(os.path.join(old_class_path, img), os.path.join(new_class_path, img))

        print(f"✅ Moved {len(images)} images for {cls} ({split}) -> {new_class_path}")

print("🎉 Dataset restructuring complete! Ready for annotation.")

✅ Moved 64014 images for wallet (train) -> /home/exh4748/ProjectTortoise/Data/wallet/images/train
✅ Moved 63973 images for watch (train) -> /home/exh4748/ProjectTortoise/Data/watch/images/train
✅ Moved 63935 images for cellphone (train) -> /home/exh4748/ProjectTortoise/Data/cellphone/images/train
✅ Moved 15986 images for wallet (val) -> /home/exh4748/ProjectTortoise/Data/wallet/images/val
✅ Moved 16027 images for watch (val) -> /home/exh4748/ProjectTortoise/Data/watch/images/val
✅ Moved 16065 images for cellphone (val) -> /home/exh4748/ProjectTortoise/Data/cellphone/images/val
✅ Moved 20000 images for wallet (test) -> /home/exh4748/ProjectTortoise/Data/wallet/images/test
✅ Moved 20000 images for watch (test) -> /home/exh4748/ProjectTortoise/Data/watch/images/test
✅ Moved 20000 images for cellphone (test) -> /home/exh4748/ProjectTortoise/Data/cellphone/images/test
🎉 Dataset restructuring complete! Ready for annotation.


In [12]:
print_directory_structure(new_dataset_path)

Data/
    wallet/
        images/
            train/
                aug_20871_02NYXWWEYUN4.jpg
                aug_90771_INQ8R7H52NW5.jpg
                aug_25679_B5NAR2N1WXD6.jpg
                aug_70094_LQP5BEN7HVJ9.jpg
                aug_39208_SKUE2L7C7CDP.jpg
                ... (64014 files)
            test/
                aug_46864_HA771L5SYFX7.jpg
                aug_66729_Z1SNN01QXALX.jpg
                aug_66863_1H10YMAMQ0GK.jpg
                aug_24490_QP55Z1M49YGE.jpg
                aug_59500_J8BUXLDGBIFY.jpg
                ... (20000 files)
            val/
                aug_18321_SB27WVSSCB1E.jpg
                aug_5242_JA0RTIRUSC28.jpg
                aug_641_B904GT456DBL.jpg
                aug_13030_90NO3LEDCO0P.jpg
                aug_6053_WDOTUV6VE1FE.jpg
                ... (15986 files)
    watch/
        images/
            train/
                aug_50499_012_0bef5f6f.jpg
                aug_34910_079_fe404f1e.jpg
                aug_31200_067_da385ca

In [13]:
# Dataset path (update this)
dataset_path = "/home/exh4748/ProjectTortoise/Data"  # Update to match your actual dataset path

# Class names (adjust if needed)
classes = ["wallet", "watch", "cellphone"]

# Splits
splits = ["train", "val", "test"]

# Target number of images per class per split (randomly selected)
min_keep = 400  # Minimum images to keep
max_keep = 600  # Maximum images to keep

# Reduce images
for cls in classes:
    for split in splits:
        image_folder = os.path.join(dataset_path, cls, "images", split)
        
        if not os.path.exists(image_folder):
            print(f"Skipping {image_folder}, folder not found.")
            continue

        # List images
        images = [f for f in os.listdir(image_folder) if f.lower().endswith((".jpg", ".png", ".jpeg"))]

        # Determine how many to keep
        keep_count = random.randint(min_keep, max_keep)
        delete_count = len(images) - keep_count

        if delete_count <= 0:
            print(f"✅ {cls} ({split}) already has {len(images)} images, no need to delete.")
            continue  # Skip if the dataset is already within range

        # Select random images to delete
        images_to_delete = random.sample(images, delete_count)

        # Delete images
        for img in images_to_delete:
            os.remove(os.path.join(image_folder, img))

        print(f"🗑️ Deleted {delete_count} images from {cls} ({split}), kept {keep_count} images.")

print("🎯 Dataset size reduction complete! Ready for auto-labeling.")

🗑️ Deleted 63511 images from wallet (train), kept 503 images.
🗑️ Deleted 15412 images from wallet (val), kept 574 images.
🗑️ Deleted 19585 images from wallet (test), kept 415 images.
🗑️ Deleted 63512 images from watch (train), kept 461 images.
🗑️ Deleted 15443 images from watch (val), kept 584 images.
🗑️ Deleted 19455 images from watch (test), kept 545 images.
🗑️ Deleted 63459 images from cellphone (train), kept 476 images.
🗑️ Deleted 15654 images from cellphone (val), kept 411 images.
🗑️ Deleted 19543 images from cellphone (test), kept 457 images.
🎯 Dataset size reduction complete! Ready for auto-labeling.


In [14]:
# Load a pre-trained YOLOv8 model
model = YOLO('yolov8n.pt')  # You can try yolov8s.pt or yolov8m.pt for better accuracy

# Define paths
dataset_path = "/home/exh4748/ProjectTortoise/Data"
splits = ["train", "val", "test"]
classes = ["wallet", "watch", "cellphone"]

# Auto-label images in each split
for cls in classes:
    for split in splits:
        image_folder = os.path.join(dataset_path, cls, "images", split)
        label_folder = os.path.join(dataset_path, cls, "labels", split)
        os.makedirs(label_folder, exist_ok=True)

        for image_path in glob.glob(os.path.join(image_folder, "*.jpg")):  # Change extension if needed
            image = cv2.imread(image_path)
            results = model(image)  # Run YOLOv8 inference

            label_file = os.path.join(label_folder, os.path.basename(image_path).replace(".jpg", ".txt"))

            with open(label_file, "w") as f:
                for r in results:
                    for box in r.boxes:
                        x_center, y_center, width, height = box.xywhn[0]  # Normalized values
                        class_id = int(box.cls)
                        f.write(f"{class_id} {x_center} {y_center} {width} {height}\n")

            print(f"Labeled: {image_path} -> {label_file}")

print("🎯 Auto-labeling complete! Labels saved in structured dataset.")

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6.25M/6.25M [00:00<00:00, 103MB/s]



0: 448x640 (no detections), 29.9ms
Speed: 3.8ms preprocess, 29.9ms inference, 8.5ms postprocess per image at shape (1, 3, 448, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/wallet/images/train/aug_76818_TCUXFBJRZ5H8.jpg -> /home/exh4748/ProjectTortoise/Data/wallet/labels/train/aug_76818_TCUXFBJRZ5H8.txt

0: 448x640 1 bird, 1 book, 4.2ms
Speed: 1.1ms preprocess, 4.2ms inference, 44.7ms postprocess per image at shape (1, 3, 448, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/wallet/images/train/aug_84129_BDNL9ELKC9UR.jpg -> /home/exh4748/ProjectTortoise/Data/wallet/labels/train/aug_84129_BDNL9ELKC9UR.txt

0: 480x640 1 book, 28.7ms
Speed: 1.1ms preprocess, 28.7ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/wallet/images/train/aug_72030_JC241V9YDMT0.jpg -> /home/exh4748/ProjectTortoise/Data/wallet/labels/train/aug_72030_JC241V9YDMT0.txt

0: 384x640 1 book, 28.7ms
Speed: 1.3ms preprocess, 28.7ms inference, 0.6ms postproc

0: 480x640 (no detections), 4.0ms
Speed: 0.8ms preprocess, 4.0ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/wallet/images/train/aug_64438_VNXQG9EX90ZT.jpg -> /home/exh4748/ProjectTortoise/Data/wallet/labels/train/aug_64438_VNXQG9EX90ZT.txt

0: 640x640 (no detections), 4.0ms
Speed: 1.7ms preprocess, 4.0ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/wallet/images/train/aug_52065_Y32DKSDUZJGB.jpg -> /home/exh4748/ProjectTortoise/Data/wallet/labels/train/aug_52065_Y32DKSDUZJGB.txt

0: 448x640 2 suitcases, 4.0ms
Speed: 0.9ms preprocess, 4.0ms inference, 0.6ms postprocess per image at shape (1, 3, 448, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/wallet/images/train/aug_93517_TEE4G7SFI6Y7.jpg -> /home/exh4748/ProjectTortoise/Data/wallet/labels/train/aug_93517_TEE4G7SFI6Y7.txt

0: 640x640 1 train, 4.1ms
Speed: 1.2ms preprocess, 4.1ms inference, 0.6ms postproce

0: 512x640 (no detections), 4.0ms
Speed: 1.0ms preprocess, 4.0ms inference, 0.2ms postprocess per image at shape (1, 3, 512, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/wallet/images/train/aug_57239_FY0U6EVX559T.jpg -> /home/exh4748/ProjectTortoise/Data/wallet/labels/train/aug_57239_FY0U6EVX559T.txt

0: 480x640 1 person, 4.0ms
Speed: 0.8ms preprocess, 4.0ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/wallet/images/train/aug_43441_TLK1QAHL87TT.jpg -> /home/exh4748/ProjectTortoise/Data/wallet/labels/train/aug_43441_TLK1QAHL87TT.txt

0: 640x640 (no detections), 4.0ms
Speed: 0.9ms preprocess, 4.0ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/wallet/images/train/aug_86638_JC0M6S79G2DX.jpg -> /home/exh4748/ProjectTortoise/Data/wallet/labels/train/aug_86638_JC0M6S79G2DX.txt

0: 480x640 1 suitcase, 4.0ms
Speed: 1.0ms preprocess, 4.0ms inference, 0.6ms postproce

0: 448x640 2 suitcases, 4.1ms
Speed: 0.9ms preprocess, 4.1ms inference, 0.6ms postprocess per image at shape (1, 3, 448, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/wallet/images/train/aug_30399_KWQTWSBIWX1S.jpg -> /home/exh4748/ProjectTortoise/Data/wallet/labels/train/aug_30399_KWQTWSBIWX1S.txt

0: 320x640 1 person, 1 hot dog, 4.1ms
Speed: 0.6ms preprocess, 4.1ms inference, 0.6ms postprocess per image at shape (1, 3, 320, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/wallet/images/train/aug_35423_Z1J1FMSEVYJP.jpg -> /home/exh4748/ProjectTortoise/Data/wallet/labels/train/aug_35423_Z1J1FMSEVYJP.txt

0: 480x640 (no detections), 4.0ms
Speed: 0.7ms preprocess, 4.0ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/wallet/images/train/aug_96148_2IL7QESGJR92.jpg -> /home/exh4748/ProjectTortoise/Data/wallet/labels/train/aug_96148_2IL7QESGJR92.txt

0: 640x640 1 tie, 4.0ms
Speed: 1.2ms preprocess, 4.0ms inference, 0.6ms postpro

...

0: 640x640 (no detections), 3.8ms
Speed: 0.8ms preprocess, 3.8ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/cellphone/images/test/aug_13452_OTWMFIAT7HMB.jpg -> /home/exh4748/ProjectTortoise/Data/cellphone/labels/test/aug_13452_OTWMFIAT7HMB.txt

0: 640x640 (no detections), 3.7ms
Speed: 0.8ms preprocess, 3.7ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/cellphone/images/test/aug_67979_YEM9XEA2ZWMU.jpg -> /home/exh4748/ProjectTortoise/Data/cellphone/labels/test/aug_67979_YEM9XEA2ZWMU.txt

0: 640x640 (no detections), 3.7ms
Speed: 0.8ms preprocess, 3.7ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/cellphone/images/test/aug_42183_HTUR8BS50HUZ.jpg -> /home/exh4748/ProjectTortoise/Data/cellphone/labels/test/aug_42183_HTUR8BS50HUZ.txt

0: 640x640 1 vase, 3.9ms
Speed: 0.8ms preprocess, 3.9ms inference, 

0: 640x640 (no detections), 3.7ms
Speed: 0.8ms preprocess, 3.7ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/cellphone/images/test/aug_43550_KYO96D0YLKIM.jpg -> /home/exh4748/ProjectTortoise/Data/cellphone/labels/test/aug_43550_KYO96D0YLKIM.txt

0: 640x640 (no detections), 3.7ms
Speed: 0.8ms preprocess, 3.7ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/cellphone/images/test/aug_20177_0QDQSD8Q4ODB.jpg -> /home/exh4748/ProjectTortoise/Data/cellphone/labels/test/aug_20177_0QDQSD8Q4ODB.txt

0: 640x640 (no detections), 3.7ms
Speed: 0.8ms preprocess, 3.7ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/cellphone/images/test/aug_36793_7A10WPWFVEF3.jpg -> /home/exh4748/ProjectTortoise/Data/cellphone/labels/test/aug_36793_7A10WPWFVEF3.txt

0: 640x640 1 cat, 3.9ms
Speed: 0.8ms preprocess, 3.9ms inference, 0

0: 640x640 (no detections), 3.7ms
Speed: 0.8ms preprocess, 3.7ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/cellphone/images/test/aug_78953_O56U42M9AK1K.jpg -> /home/exh4748/ProjectTortoise/Data/cellphone/labels/test/aug_78953_O56U42M9AK1K.txt

0: 640x640 (no detections), 3.7ms
Speed: 0.8ms preprocess, 3.7ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/cellphone/images/test/aug_51582_XPMB2H4Z12HK.jpg -> /home/exh4748/ProjectTortoise/Data/cellphone/labels/test/aug_51582_XPMB2H4Z12HK.txt

0: 640x640 2 persons, 3.7ms
Speed: 0.8ms preprocess, 3.7ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/cellphone/images/test/aug_90275_NL3X0UCQQXQQ.jpg -> /home/exh4748/ProjectTortoise/Data/cellphone/labels/test/aug_90275_NL3X0UCQQXQQ.txt

0: 640x640 (no detections), 3.7ms
Speed: 0.8ms preprocess, 3.7ms inferenc

0: 640x640 (no detections), 3.7ms
Speed: 0.8ms preprocess, 3.7ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/cellphone/images/test/aug_98353_1SXHMG4FZ9P0.jpg -> /home/exh4748/ProjectTortoise/Data/cellphone/labels/test/aug_98353_1SXHMG4FZ9P0.txt

0: 640x640 (no detections), 3.7ms
Speed: 0.8ms preprocess, 3.7ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/cellphone/images/test/aug_22321_Q5512GGZ5AS0.jpg -> /home/exh4748/ProjectTortoise/Data/cellphone/labels/test/aug_22321_Q5512GGZ5AS0.txt

0: 640x640 (no detections), 3.8ms
Speed: 0.8ms preprocess, 3.8ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/cellphone/images/test/aug_91159_CUYE9PGQ29XR.jpg -> /home/exh4748/ProjectTortoise/Data/cellphone/labels/test/aug_91159_CUYE9PGQ29XR.txt

0: 640x640 (no detections), 3.9ms
Speed: 0.8ms preprocess, 3.9ms in

0: 640x640 (no detections), 3.9ms
Speed: 0.8ms preprocess, 3.9ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/cellphone/images/test/aug_17919_7SEAPC331IUM.jpg -> /home/exh4748/ProjectTortoise/Data/cellphone/labels/test/aug_17919_7SEAPC331IUM.txt

0: 640x640 (no detections), 3.9ms
Speed: 0.8ms preprocess, 3.9ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/cellphone/images/test/aug_83428_7I0ZKD4G7P2H.jpg -> /home/exh4748/ProjectTortoise/Data/cellphone/labels/test/aug_83428_7I0ZKD4G7P2H.txt

0: 640x640 (no detections), 3.9ms
Speed: 0.8ms preprocess, 3.9ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 640)
Labeled: /home/exh4748/ProjectTortoise/Data/cellphone/images/test/aug_72716_Q0GSH2N3O0AK.jpg -> /home/exh4748/ProjectTortoise/Data/cellphone/labels/test/aug_72716_Q0GSH2N3O0AK.txt

0: 640x640 (no detections), 4.0ms
Speed: 0.8ms preprocess, 4.0ms in

In [15]:
dataset_path = "/home/exh4748/ProjectTortoise/Data"  # Update with your dataset path

for root, _, files in os.walk(dataset_path):
    for file in files:
        if file.endswith(".txt"):
            label_path = os.path.join(root, file)
            if os.stat(label_path).st_size == 0:  # Check if file is empty
                os.remove(label_path)
                print(f"🗑️ Deleted empty label: {label_path}")

print("✅ Removed all empty label files.")

🗑️ Deleted empty label: /home/exh4748/ProjectTortoise/Data/wallet/labels/train/aug_37885_5K5AS29D24MM.txt
🗑️ Deleted empty label: /home/exh4748/ProjectTortoise/Data/wallet/labels/train/aug_93762_XDV1X3L7I1LA.txt
🗑️ Deleted empty label: /home/exh4748/ProjectTortoise/Data/wallet/labels/train/aug_38416_F3C45ZBS9UQH.txt
🗑️ Deleted empty label: /home/exh4748/ProjectTortoise/Data/wallet/labels/train/aug_31435_283FFFR6PUDX.txt
🗑️ Deleted empty label: /home/exh4748/ProjectTortoise/Data/wallet/labels/train/aug_64694_ZYT1UHKSES0W.txt
🗑️ Deleted empty label: /home/exh4748/ProjectTortoise/Data/wallet/labels/train/aug_51028_H2NM6GXP2GMB.txt
🗑️ Deleted empty label: /home/exh4748/ProjectTortoise/Data/wallet/labels/train/aug_48302_58G5T6OGUUFL.txt
🗑️ Deleted empty label: /home/exh4748/ProjectTortoise/Data/wallet/labels/train/aug_97865_W4G8BYY5OEGK.txt
🗑️ Deleted empty label: /home/exh4748/ProjectTortoise/Data/wallet/labels/train/aug_30768_R0JQ2S0C8ZL4.txt
🗑️ Deleted empty label: /home/exh4748/ProjectT

In [16]:
dataset_path = "/home/exh4748/ProjectTortoise/Data"  # Update with your dataset path
splits = ["train", "val", "test"]
classes = ["wallet", "watch", "cellphone"]

for cls in classes:
    for split in splits:
        image_folder = os.path.join(dataset_path, cls, "images", split)
        label_folder = os.path.join(dataset_path, cls, "labels", split)

        images = [f for f in os.listdir(image_folder) if f.endswith((".jpg", ".png", ".jpeg"))]

        for img in images:
            label_file = os.path.join(label_folder, img.replace(".jpg", ".txt").replace(".png", ".txt"))
            if not os.path.exists(label_file):
                os.remove(os.path.join(image_folder, img))  # Remove image if label does not exist
                print(f"🗑️ Deleted unlabeled image: {img}")

print("✅ Removed all images without labels.")

🗑️ Deleted unlabeled image: aug_76818_TCUXFBJRZ5H8.jpg
🗑️ Deleted unlabeled image: aug_85505_YZHAS954PTLM.jpg
🗑️ Deleted unlabeled image: aug_21381_940P3KIEA30Z.jpg
🗑️ Deleted unlabeled image: aug_32060_DCRRM2EAH74G.jpg
🗑️ Deleted unlabeled image: aug_54296_0L9SQU1M0AGQ.jpg
🗑️ Deleted unlabeled image: aug_33625_412NIGUHAG6R.jpg
🗑️ Deleted unlabeled image: aug_97865_W4G8BYY5OEGK.jpg
🗑️ Deleted unlabeled image: aug_64694_ZYT1UHKSES0W.jpg
🗑️ Deleted unlabeled image: aug_30467_LYLOIKIFFKS5.jpg
🗑️ Deleted unlabeled image: aug_25800_DD3X9XXAK31X.jpg
🗑️ Deleted unlabeled image: aug_64438_VNXQG9EX90ZT.jpg
🗑️ Deleted unlabeled image: aug_52065_Y32DKSDUZJGB.jpg
🗑️ Deleted unlabeled image: aug_52265_1NJE54RCXUEG.jpg
🗑️ Deleted unlabeled image: aug_68512_U2YYQQYYHCIN.jpg
🗑️ Deleted unlabeled image: aug_30344_K7Y4I6Z68MU3.jpg
🗑️ Deleted unlabeled image: aug_86575_ICRB7DX16KTU.jpg
🗑️ Deleted unlabeled image: aug_48943_H51S93Q7LR0Q.jpg
🗑️ Deleted unlabeled image: aug_64207_RVBUIBR2HMR4.jpg
🗑️ Deleted

In [18]:
# Base dataset path
dataset_path = "/home/exh4748/ProjectTortoise/Data"  # Update this path

# Define classes
classes = ["wallet", "watch", "cellphone"]

# New YOLOv8 structure
yolo_structure = {
    "images": {"train": [], "val": [], "test": []},
    "labels": {"train": [], "val": [], "test": []}
}

# Create new dataset structure
for category in ["train", "val", "test"]:
    os.makedirs(os.path.join(dataset_path, "images", category), exist_ok=True)
    os.makedirs(os.path.join(dataset_path, "labels", category), exist_ok=True)

# Move images and labels to new structure
for cls in classes:
    for split in ["train", "val", "test"]:
        old_img_path = os.path.join(dataset_path, cls, "images", split)
        old_lbl_path = os.path.join(dataset_path, cls, "labels", split)

        new_img_path = os.path.join(dataset_path, "images", split)
        new_lbl_path = os.path.join(dataset_path, "labels", split)

        if os.path.exists(old_img_path):
            for img in os.listdir(old_img_path):
                shutil.move(os.path.join(old_img_path, img), os.path.join(new_img_path, img))

        if os.path.exists(old_lbl_path):
            for lbl in os.listdir(old_lbl_path):
                shutil.move(os.path.join(old_lbl_path, lbl), os.path.join(new_lbl_path, lbl))

print("✅ Dataset merged into YOLOv8 format!")

✅ Dataset merged into YOLOv8 format!


# Model Training

In [20]:
# Load YOLOv8 model
model = YOLO("yolov8n.pt")  # You can use yolov8s.pt for better accuracy

# Train model
model.train(
    data="/home/exh4748/ProjectTortoise/Beta/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device="cuda"  # Use "cpu" if no GPU
)

New https://pypi.org/project/ultralytics/8.3.75 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.74 🚀 Python-3.9.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 45486MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/home/exh4748/ProjectTortoise/Beta/data.yaml, epochs=50, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=cuda, workers=8, project=None, name=train3, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=Fal

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5.35M/5.35M [00:00<00:00, 103MB/s]


AMP: checks passed ✅


train: Scanning /home/exh4748/ProjectTortoise/Data/labels/train... 850 images, 0 backgrounds, 725 corrupt: 100%|██████████| 850/850 [00:00<00:00, 3385.33it/

train: WARNING ⚠️ /home/exh4748/ProjectTortoise/Data/images/train/aug_20001_023_5f9b2835.jpg: ignoring corrupt image/label: Label class 74 exceeds dataset class count 3. Possible class labels are 0-2
train: WARNING ⚠️ /home/exh4748/ProjectTortoise/Data/images/train/aug_20022_LVIXIF4DJAQ8.jpg: ignoring corrupt image/label: Label class 7 exceeds dataset class count 3. Possible class labels are 0-2
train: WARNING ⚠️ /home/exh4748/ProjectTortoise/Data/images/train/aug_20091_N07YKKO3957Q.jpg: ignoring corrupt image/label: Label class 28 exceeds dataset class count 3. Possible class labels are 0-2
train: WARNING ⚠️ /home/exh4748/ProjectTortoise/Data/images/train/aug_20135_038_63e657f6.jpg: ignoring corrupt image/label: Label class 74 exceeds dataset class count 3. Possible class labels are 0-2
train: WARNING ⚠️ /home/exh4748/ProjectTortoise/Data/images/train/aug_20216_OXNW3Q5IAKTS.jpg: ignoring corrupt image/label: Label class 28 exceeds dataset class count 3. Possible class labels are 0-2
t

...

                   all        116        147      0.378      0.328      0.376      0.299



50 epochs completed in 0.014 hours.
Optimizer stripped from runs/detect/train3/weights/last.pt, 6.2MB
Optimizer stripped from runs/detect/train3/weights/best.pt, 6.2MB

Validating runs/detect/train3/weights/best.pt...
Ultralytics 8.3.74 🚀 Python-3.9.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 45486MiB)
Model summary (fused): 168 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:00<00:00,  9.89it/s]


                   all        116        147      0.396      0.324      0.388      0.303
                wallet        115        145      0.792      0.648      0.686       0.55
             cellphone          2          2          0          0     0.0905     0.0561
Speed: 0.1ms preprocess, 0.3ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to runs/detect/train3


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f9b749a13a0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

# Evaluate Model

In [46]:
# Update with the actual path you found
model_path = "runs/detect/train3/weights/best.pt"  # Replace with correct folder

# Load the trained model
model = YOLO(model_path)  # Path to the best model

# Run validation to get performance metrics
metrics = model.val()

Ultralytics 8.3.74 🚀 Python-3.9.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 45486MiB)
Model summary (fused): 168 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs


val: Scanning /home/exh4748/ProjectTortoise/Data/labels/val.cache... 954 images, 0 backgrounds, 838 corrupt: 100%|██████████| 954/954 [00:00<?, ?it/s]

val: WARNING ⚠️ /home/exh4748/ProjectTortoise/Data/images/val/aug_10010_XWYS33P3UH46.jpg: ignoring corrupt image/label: Label class 62 exceeds dataset class count 3. Possible class labels are 0-2
val: WARNING ⚠️ /home/exh4748/ProjectTortoise/Data/images/val/aug_10026_T6EJGBF7E8I3.jpg: ignoring corrupt image/label: Label class 55 exceeds dataset class count 3. Possible class labels are 0-2
val: WARNING ⚠️ /home/exh4748/ProjectTortoise/Data/images/val/aug_10030_115_d5146709.jpg: ignoring corrupt image/label: Label class 74 exceeds dataset class count 3. Possible class labels are 0-2
val: WARNING ⚠️ /home/exh4748/ProjectTortoise/Data/images/val/aug_10068_TYGKB5T8A9Q1.jpg: ignoring corrupt image/label: Label class 73 exceeds dataset class count 3. Possible class labels are 0-2
val: WARNING ⚠️ /home/exh4748/ProjectTortoise/Data/images/val/aug_1009_113_1dc32d4e.jpg: ignoring corrupt image/label: Label class 74 exceeds dataset class count 3. Possible class labels are 0-2
val: WARNING ⚠️ /home


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:00<00:00, 12.98it/s]


                   all        116        147      0.399      0.321      0.391      0.304
                wallet        115        145      0.799      0.641      0.691      0.552
             cellphone          2          2          0          0     0.0905     0.0569
Speed: 0.6ms preprocess, 0.5ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to runs/detect/val14


In [47]:
# Extract key validation metrics
mAP_50 = metrics.box.map50  # Mean Average Precision at IoU 0.5
mAP_50_95 = metrics.box.map  # Mean Average Precision at IoU 0.5:0.95
precision = metrics.box.mp  # Mean Precision
recall = metrics.box.mr  # Mean Recall
f1_score = metrics.box.f1.mean()  # Mean F1-score

In [48]:
# Print extracted metrics
print(f"📊 YOLOv8 Training & Validation Metrics")
print(f"------------------------------------")
print(f"mAP@50: {mAP_50:.4f}")
print(f"mAP@50-95: {mAP_50_95:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1_score:.4f}")
print(f"------------------------------------")

📊 YOLOv8 Training & Validation Metrics
------------------------------------
mAP@50: 0.3905
mAP@50-95: 0.3044
Precision: 0.3994
Recall: 0.3207
F1-Score: 0.3557
------------------------------------
